#  Phil Job Applier Agent - Rachel Briskman & Sanila Chowdhury (Daedalus Devs)

## Team Report #1 (03/24) - Created a function that takes in user's resume and parse it to output the extracted info & uses Skyvern to see how the library works for filling in the fields of a job website

<img src="AI_AGENT_WORKFLOW.png" width="700">

### Installs and Imports

In [ ]:

%pip install chromadb
# %pip install "skyvern==1.0.24"
%pip install skyvern
# %playwright install --with-deps chromium
# %playwright install chromium
%pip install langchain langchain-huggingface
%pip install python-dotenv

print ("Done installing")

In [ ]:
#installing dependencies for browser-use (an alternative to skyvern)
%pip install browser-use playwright
%pip install -U langchain-google-genai
%pip install -U browser-use
# %playwright install chromium

In [ ]:
import os
from dotenv import load_dotenv
import platform
import sys


#Sanila needs to set the working directory explicity (macOS) but it's not necessary on Windows or WSL.
#added this macro so we don't have to manually comment/uncommment os.chdir() every time.

IS_MAC = platform.system() == "Darwin"

if IS_MAC:
    os.chdir("/Users/sanilachowdhury/Desktop/ai_agents")
    print("Running on macOS")
else:
    print("Not running on macOS, current directory:", os.getcwd())


load_dotenv()


gemini_key = os.getenv("GEMINI_API_KEY")
skyvern_key = os.getenv("SKYVERN_API_KEY")
phil_key = os.getenv("PHIL-KEY")


os.environ["ENABLE_AUTH"] = "false"

# from skyvern.forge.sdk import SkyvernForge
from skyvern import Skyvern
import nest_asyncio
import asyncio

# Apply nest_asyncio to allow nested event loops in Colab
nest_asyncio.apply()

# 2. Map it to the environment variables Skyvern/LiteLLM looks for
os.environ["GEMINI_API_KEY"] = gemini_key
# os.environ["GOOGLE_API_KEY"] = gemini_key  # Some versions prefer this
# Optional: Force Skyvern to use a specific model
os.environ["LLM_KEY"] = "gemini/gemini-1.5-flash"
print(f"Key check: {'FOUND' if os.environ.get('GEMINI_API_KEY') else 'MISSING'}")
# print("Keys loaded and environment configured.")

### A website we're feeding into our agent to fill in the field (this website is used for testing purpose)

In [ ]:
url = "https://demo.applitools.com/"

### Agent takes in the filepath where the user's resume is stored and we prompt the agent to extract the info in a JSON that easy for both LLM and humans to read

In [ ]:
#NOTE: This uses browser-use instead of skyvern. This cell works but it does not promp the user for addtional info when needed. See next cell.
import os
import asyncio
from dotenv import load_dotenv
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm import ChatGoogle

load_dotenv()

url = "https://demo.applitools.com/"

async def run_locally():
    llm = ChatGoogle(
        model="gemini-2.5-flash-lite",
        api_key=os.getenv("GEMINI_API_KEY")
    )

    browser_profile = BrowserProfile(headless=False, keep_alive=True)
    browser_session = BrowserSession(browser_profile=browser_profile)

    agent = Agent(
        task=f"Go to {url} and fill in 'test_user' and 'password123' BUT DO NOT PRESS SUBMIT OR LOGIN. Return DONE when complete",
        llm=llm,
        browser_session=browser_session,
    )

    result = await agent.run()
    print(result)

await run_locally()

In [ ]:
from browser_use import Agent, BrowserProfile, BrowserSession
from browser_use.llm import ChatGoogle

async def run_job_application():
    llm = ChatGoogle(model="gemini-2.5-flash-lite", api_key=os.getenv("GEMINI_API_KEY"))
    
    browser_profile = BrowserProfile(headless=False, keep_alive=True)
    browser_session = BrowserSession(browser_profile=browser_profile)

    # Phase 1: fill what we know
    agent = Agent(
        task=f"""Go to {url} and fill in the username with a_cool_username_123. 
        If you encounter fields you don't have info for, stop and list what's missing. 
        Do NOT submit.""",
        llm=llm,
        browser_session=browser_session,
        use_vision=False,  # reduces tokens significantly
    )
    result = await agent.run()
    print("Agent stopped. Result:", result)


    #NOTE: Might be better to put this in a second cell or loop this?
    # Phase 2: ask user for missing info
    missing = input("What additional info do you want to provide? ")

    # Phase 3: continue with same browser session (no reload, no extra screenshots)
    agent2 = Agent(
        task=f"Fill in the remaining fields with: {missing}. Do NOT submit.",
        llm=llm,
        browser_session=browser_session,  # reuse same session!
    )
    result2 = await agent2.run()
    print("Done:", result2)

await run_job_application()

In [ ]:
from skyvern import Skyvern
import asyncio
import nest_asyncio
from google import genai
# from google.colab import userdata

nest_asyncio.apply()

client_genai = genai.Client(api_key=gemini_key)

# this takes in user's resume and parse and LLM will output the info it extracted to ensure accuracy
def parse_pdf_with_gemini(file_path, prompt="Please parse this resume and return its contents in a structured JSON that is easy for both LLM and humans to read."):
    """
    Uploads a PDF to Gemini and returns the model's analysis.
    """
    # upload user's resume
    print(f"Uploading {file_path}...")
    uploaded_file = client_genai.files.upload(
        file=file_path,
        config={"display_name": "User_PDF"}
    )

   # feeds LLM the prompt and uploaded resume PDF
    response = client_genai.models.generate_content(
        # model="gemini-2.0-flash",
        model="gemini-2.5-flash-lite",
        contents=[uploaded_file, prompt]
    )

    return response.text

# call the function to store the extracted info and print it
result = parse_pdf_with_gemini("content/jakes-resume.pdf")
print(result)




In [ ]:
def verify_resume_info(extracted_info):
    current_info = extracted_info

    while True:
        print("\n" + "="*10)
        print("EXTRACTED RESUME INFO:")
        print("="*10)
        print(current_info)
        print("="*10)

        user_confirm = input("\nIs this information correct? (y/n): ").strip().lower()

        if user_confirm == "y":
            print("\nGreat! Moving forward with this information.")
            return current_info
        else:
            correction = input("\nWhat needs to be corrected or added? Please describe: ").strip()

            print("\nUpdating extracted info...")
            response = client_genai.models.generate_content(
                model="gemini-2.5-flash-lite",
                contents=[
                    f"""Here is the currently extracted resume information:
{current_info}

The user wants the following correction or addition:
{correction}

Please update the extracted resume JSON accordingly and return the full updated JSON."""
                ]
            )
            current_info = response.text
            print("\nInfo updated! Let's review it again.")


# call the function to store the extracted info
result = parse_pdf_with_gemini("content/jakes-resume.pdf")

# run verify_resume_info to verify and correct
final_info = verify_resume_info(result)

print("\nFinal verified resume info:")
print(final_info)

### Setting up ChromaDB which will store the correct parsed info of user's resume

In [ ]:
import chromadb
import os
import hashlib
from dotenv import load_dotenv

# debug - check if values are loaded
print(f"CHROMA_HOST: {os.getenv('CHROMA_HOST')}")
print(f"CHROMA_TENANT: {os.getenv('CHROMA_TENANT')}")
print(f"CHROMA_DATABASE: {os.getenv('CHROMA_DATABASE')}")
print(f"CHROMA_API_KEY: {'FOUND' if os.getenv('CHROMA_API_KEY') else 'MISSING'}")

chroma_client = chromadb.HttpClient(
    ssl=True,
    host=os.getenv("CHROMA_HOST"),
    tenant=os.getenv("CHROMA_TENANT"),
    database=os.getenv("CHROMA_DATABASE"),
    headers={"x-chroma-token": os.getenv("CHROMA_API_KEY")}
)
collection = chroma_client.get_or_create_collection(name="resumes")
print(f"Connected to ChromaDB! Total resumes stored: {collection.count()}")

def store_resume_in_chromadb(resume_text, file_path):
    doc_id = hashlib.md5(file_path.encode()).hexdigest()
    collection.upsert(
        documents=[resume_text],
        ids=[doc_id],
        metadatas=[{"file_path": file_path}]
    )
    print(f"Resume stored! ID: {doc_id}")
    return doc_id

# store the verified resume
doc_id = store_resume_in_chromadb(final_info, "content/jakes-resume.pdf")
print("Resume stored successfully!")

### Created a popup window of the job website that user will feed into Phil

## Plans for spring break:
* Implementing a function where our agent to fill in the fields from the job website using Skyvern (we have started this process but currently experiencing errors)
* Implementing a loop for our agent to handle when it comes to short-answer response
  * Have a prompt that will tell the LLM to generate answer based on the context given and the info it extracted from user's resume
  * Have a loop where as long as the answer is too vague, keep asking user for more info and implement a function for LLM to do web search (depending on the short answer question)
  * Have LLM generate a short answer response and fill out the field with Skyvern
* IF WE FIND TIME -> Have an evalulation where user can give the LLM feedback at the end of the overall performance and store that in ChromaDB to help Phil improve